In [8]:
import pandas as pd 

In [3]:
df = pd.read_csv('Train2.csv')

In [7]:
df.sample(5) 

,url,Phish?
70463,http://onebone.square.site,1
642,http://apsolutnadestrukcija.com/jaoo-pazi-ogro...,0
57304,http://docs.google.com/presentation/d/e/2PACX-...,1
82664,http://askgriff.com,1
110989,http://www.senshi-akademie.com/t876-die-prinze...,0


In [9]:
pip install tldextract


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [tldextract]
Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
import re
import tldextract
import math
from urllib.parse import urlparse
from collections import Counter

def shannon_entropy(string):
    if not string:
        return 0
    prob = [float(string.count(c)) / len(string) for c in dict.fromkeys(list(string))]
    entropy = -sum([p * math.log2(p) for p in prob])
    return entropy

def has_ip(hostname):
    pattern = r'(\d{1,3}\.){3}\d{1,3}'
    return 1 if re.search(pattern, hostname) else 0

def count_digits(s):
    return sum(c.isdigit() for c in s)

def count_letters(s):
    return sum(c.isalpha() for c in s)

def count_special_chars(s):
    # Counts characters that are not alphanumeric
    return len(re.findall(r'[^a-zA-Z0-9]', s))

def count_words(s):
    words = re.split(r'[\W_]+', s)
    words = [w for w in words if w]
    return len(words)

def is_shortened(url):
    pattern = r'bit\.ly|goo\.gl|shorte\.st|go2l\.ink|x\.co|ow\.ly|t\.co|tinyurl|is\.gd|cli\.gs|tr\.im'
    return 1 if re.search(pattern, url.lower()) else 0

def keyword_count(url):
    suspicious_keywords = [
        'login', 'secure', 'update', 'bank', 'account',
        'verify', 'paypal', 'signin', 'confirm', 'free',
        'webscr', 'ebay', 'amazon', 'wallet', 'bonus'
    ]
    return sum(word in url.lower() for word in suspicious_keywords)


def extract_features(url):
    if not isinstance(url, str) or len(url) == 0:
        return None

    parsed = urlparse(url)
    ext = tldextract.extract(url)
    
    hostname = parsed.netloc
    domain = ext.domain
    subdomain = ext.subdomain
    path = parsed.path
    url_len = len(url)
    
    features = {}
    
    features['url_length'] = url_len
    features['domain_length'] = len(domain)
    features['subdomain_length'] = len(subdomain)
    features['path_length'] = len(path)
    
    features['dot_count'] = url.count('.')
    features['dash_count'] = url.count('-')
    features['underscore_count'] = url.count('_')
    features['slash_count'] = url.count('/')
    features['question_count'] = url.count('?')
    features['equal_count'] = url.count('=')
    features['at_count'] = url.count('@')
    features['hostname_dot_count'] = hostname.count('.')
    
    features['digit_count'] = count_digits(url)
    features['letter_count'] = count_letters(url)
    features['special_char_count'] = count_special_chars(url)
    
    features['digit_ratio'] = features['digit_count'] / url_len if url_len > 0 else 0
    features['letter_ratio'] = features['letter_count'] / url_len if url_len > 0 else 0
    
    features['entropy'] = shannon_entropy(url)
    features['has_ip'] = has_ip(hostname) # Checked against hostname specifically
    features['is_shortened'] = is_shortened(url)
    
    # Check if 'www' appears anywhere except the very start
    features['bad_www'] = 1 if 'www' in url.lower() and not url.lower().startswith('http://www') else 0
    
    # Check if a common TLD is hidden in the path (e.g., example.com/paypal.com/)
    tld_in_path_pattern = r'\.com/|\.net/|\.org/|\.edu/|\.gov/'
    features['tld_in_path'] = 1 if re.search(tld_in_path_pattern, path.lower()) else 0
    
    features['word_count'] = count_words(url)
    features['keyword_count'] = keyword_count(url)
    
    suspicious_tlds = ['tk', 'ml', 'ga', 'cf', 'gq', 'xyz', 'top', 'pw']
    features['suspicious_tld'] = 1 if ext.suffix in suspicious_tlds else 0
    
    return features


def build_feature_dataframe(df, url_column):
    
    feature_list = df[url_column].apply(lambda x: extract_features(x))
    
    feature_df = pd.DataFrame(feature_list.tolist())
    
    feature_df = feature_df.fillna(0)
    
    return feature_df

In [12]:
feature_matrix = build_feature_dataframe(df, 'url')


Starting feature extraction for 111401 URLs...
Feature extraction complete.


In [13]:
feature_matrix

,url_length,domain_length,subdomain_length,path_length,dot_count,dash_count,underscore_count,slash_count,question_count,equal_count,...,digit_ratio,letter_ratio,entropy,has_ip,is_shortened,bad_www,tld_in_path,word_count,keyword_count,suspicious_tld
0,36,15,0,10,1,0,0,4,0,0,...,0.000000,0.833333,3.993133,0,0,0,0,5,0,0
1,45,4,8,18,4,0,1,3,0,0,...,0.022222,0.777778,4.402530,0,0,0,0,8,0,0
2,20,3,0,7,1,1,0,3,0,0,...,0.050000,0.650000,3.746439,0,0,0,0,5,0,0
3,60,16,3,20,3,1,0,4,1,1,...,0.033333,0.766667,4.460324,0,0,0,0,11,0,0
4,174,6,4,108,2,1,0,7,1,3,...,0.109195,0.781609,5.574198,0,0,0,0,18,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111396,147,11,0,125,1,16,0,6,0,0,...,0.054422,0.782313,4.445113,0,0,0,0,23,0,0
111397,32,7,3,8,3,0,0,3,0,0,...,0.000000,0.781250,3.905639,0,0,0,0,6,0,0
111398,50,6,3,29,2,1,0,5,0,0,...,0.040000,0.780000,4.351272,0,0,0,0,8,0,0
111399,20,6,0,3,1,0,0,3,0,0,...,0.000000,0.750000,3.608695,0,0,0,0,4,0,1


In [18]:
final_data = pd.concat([feature_matrix, df['Phish?']], axis=1)

In [19]:
final_data 

,url_length,domain_length,subdomain_length,path_length,dot_count,dash_count,underscore_count,slash_count,question_count,equal_count,...,letter_ratio,entropy,has_ip,is_shortened,bad_www,tld_in_path,word_count,keyword_count,suspicious_tld,Phish?
0,36,15,0,10,1,0,0,4,0,0,...,0.833333,3.993133,0,0,0,0,5,0,0,0
1,45,4,8,18,4,0,1,3,0,0,...,0.777778,4.402530,0,0,0,0,8,0,0,0
2,20,3,0,7,1,1,0,3,0,0,...,0.650000,3.746439,0,0,0,0,5,0,0,1
3,60,16,3,20,3,1,0,4,1,1,...,0.766667,4.460324,0,0,0,0,11,0,0,0
4,174,6,4,108,2,1,0,7,1,3,...,0.781609,5.574198,0,0,0,0,18,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111396,147,11,0,125,1,16,0,6,0,0,...,0.782313,4.445113,0,0,0,0,23,0,0,0
111397,32,7,3,8,3,0,0,3,0,0,...,0.781250,3.905639,0,0,0,0,6,0,0,0
111398,50,6,3,29,2,1,0,5,0,0,...,0.780000,4.351272,0,0,0,0,8,0,0,0
111399,20,6,0,3,1,0,0,3,0,0,...,0.750000,3.608695,0,0,0,0,4,0,1,1


In [28]:
final_data[(final_data['is_shortened'] == 1) & (final_data['Phish?']== 1 )].size

111670

In [29]:
final_data[(final_data['is_shortened'] == 1) & (final_data['Phish?']== 0 )].size

199732

In [31]:
final_data.to_csv('Final.csv' , index = False )